## **Missing Values Imputation**

### 1. Dataset Awal

| No | IPK | PO     | JML |
| -- | --- | ------ | --- |
| 1  | 2   | 200000 | 2   |
| 2  | 3   | 300000 | 3   |
| 3  | 4   | 200000 | 2   |
| 4  | 2   | 200000 | 3   |
| 5  | 3   | 300000 | 2   |
| 6  | 4   | 400000 | 3   |
| 7  | 2   | 300000 | ?   |

Nilai JML pada data ke-7 merupakan **missing value** sehingga perlu dilakukan imputation menggunakan metode **WKNN**.

### 2. Normalisasi Data (MIN-MAX)

Rumus normalisasi min-max:

$$
X' = \frac{X - X_{min}}{X_{max} - X_{min}}
$$

Keterangan

- $X$ = nilai data asli  
- $X_{min}$ = nilai minimum data  
- $X_{max}$ = nilai maksimum data  
- $X'$ = nilai hasil normalisasi

**Nilai Minimum dan Maksimum**

| Atribut | Min    | Max    |
| ------- | ------ | ------ |
| IPK     | 2      | 4      |
| PO      | 200000 | 400000 |
| JML     | 2      | 3      |

**Hasil Normalisasi**

| IPK | PO  | JML |
| --- | --- | --- |
| 0   | 0   | 0   |
| 0.5 | 0.5 | 1   |
| 1   | 0   | 0   |
| 0   | 0   | 1   |
| 0.5 | 0.5 | 0   |
| 1   | 1   | 1   |
| 0   | 0.5 | ?   |

Data ke-7 yang akan diprediksi:

IPK = 0

PO = 0.5

### 3. Menghitung WKNN (Manual)

Rumus jarak **Euclidean Distance**:

$$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

**Perhitungan jarak**

| Data | Perhitungan            | Jarak | JML |
| ---- | ---------------------- | ----- | --- |
| 1    | √((0−0)²+(0−0.5)²)     | 0.5   | 2   |
| 2    | √((0.5−0)²+(0.5−0.5)²) | 0.5   | 3   |
| 3    | √((1−0)²+(0−0.5)²)     | 1.118 | 2   |
| 4    | √((0−0)²+(0−0.5)²)     | 0.5   | 3   |
| 5    | √((0.5−0)²+(0.5−0.5)²) | 0.5   | 2   |
| 6    | √((1−0)²+(1−0.5)²)     | 1.118 | 3   |


**Ambil K terdekat (K=3)**

| Data | Jarak | JML |
| ---- | ----- | --- |
| 1    | 0.5   | 2   |
| 2    | 0.5   | 3   |
| 4    | 0.5   | 3   |

**Hitung Bobot WKNN**

Rumus bobot:

$$
W = \frac{1}{d}
$$

| Data | Bobot | JML |
| ---- | ----- | --- |
| 1    | 2     | 2   |
| 2    | 2     | 3   |
| 4    | 2     | 3   |


### 4. Hasil Prediksi Missing Value

$$
\frac{(2 \times 2) + (2 \times 3) + (2 \times 3)}{2 + 2 + 2}
$$

$$
= \frac{4 + 6 + 6}{6}
$$

$$
= \frac{16}{6}
$$

$$
= 2.67
$$

$$
\text{Dibulatkan: } JML = 3
$$

Dataset setelah imputasi:

| No | IPK | PO     | JML |
| -- | --- | ------ | --- |
| 1  | 2   | 200000 | 2   |
| 2  | 3   | 300000 | 3   |
| 3  | 4   | 200000 | 2   |
| 4  | 2   | 200000 | 3   |
| 5  | 3   | 300000 | 2   |
| 6  | 4   | 400000 | 3   |
| 7  | 2   | 300000 | 3   |


### 5. Implementasi Code Python WKNN

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor

# dataset
data = {
    'IPK':[2,3,4,2,3,4,2],
    'PO':[200000,300000,200000,200000,300000,400000,300000],
    'JML':[2,3,2,3,2,3,np.nan]
}

df = pd.DataFrame(data)

# normalisasi Min-Max
scaler = MinMaxScaler()
df[['IPK','PO']] = scaler.fit_transform(df[['IPK','PO']])

# pisahkan data training dan testing
train = df[df['JML'].notna()]
test = df[df['JML'].isna()]

X_train = train[['IPK','PO']]
y_train = train['JML']

# WKNN
knn = KNeighborsRegressor(n_neighbors=3, weights='distance')
knn.fit(X_train, y_train)

# prediksi missing value
pred = knn.predict(test[['IPK','PO']])

print("Prediksi JML:", round(pred[0],2))
print("Hasil pembulatan:", round(pred[0]))

Prediksi JML: 2.33
Hasil pembulatan: 2


**Catatan Penjelasan:**

Pada perhitungan manual menggunakan metode **WKNN** dengan (k = 3), diperoleh hasil **JML = 3**. Hal ini karena tiga data tetangga yang dipilih secara manual memiliki nilai JML **2, 3, dan 3**, sehingga hasil rata-ratanya adalah **2.67** dan kemudian dibulatkan menjadi **3**.

Namun ketika dihitung menggunakan **kode Python**, hasil yang diperoleh adalah **2**. Hal ini terjadi karena program secara otomatis memilih tiga data tetangga terdekat berdasarkan urutan data pada dataset. Karena terdapat beberapa data yang memiliki **jarak yang sama**, kombinasi tetangga yang dipilih oleh program berbeda dengan yang dipilih pada perhitungan manual.

Akibatnya, nilai yang dihitung oleh program menjadi sekitar **2.33** dan setelah dibulatkan hasilnya adalah **2**.

Jadi perbedaan hasil ini terjadi karena **cara pemilihan tetangga ketika jaraknya sama**, bukan karena perhitungan manual atau kode yang salah.
